# Lesson 0b: RL Setup and Practical Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powell-clark/reinforcement-learning/blob/main/notebooks/0b_rl_setup_practical.ipynb)

**Author**: Powell-Clark Limited  
**Course**: Reinforcement Learning from First Principles  
**Lesson**: 0b - RL Setup and Practical Tools (Practical)  
**Updated**: 2025 - Aligned with elite university standards

---

## Learning Objectives

By the end of this notebook, you will master:
1. **Gymnasium API** - The standard interface for RL environments
2. **Creating custom environments** - Build your own RL problems
3. **Environment wrappers** - Modify behavior without changing environment code
4. **Visualization techniques** - Monitor training and debug agents
5. **Debugging RL agents** - Common pitfalls and how to fix them
6. **Best practices** - Professional RL development workflow

## Prerequisites

- Lesson 0a: Introduction to Reinforcement Learning (Theory)
- Basic Python programming
- Familiarity with NumPy

## Table of Contents

1. [Gymnasium API Deep Dive](#1-gymnasium-api)
2. [Exploring Built-in Environments](#2-built-in-environments)
3. [Creating Custom Environments](#3-custom-environments)
4. [Environment Wrappers](#4-environment-wrappers)
5. [Visualization and Monitoring](#5-visualization)
6. [Debugging RL Agents](#6-debugging)
7. [Best Practices](#7-best-practices)
8. [Summary](#8-summary)

In [ ]:
# Setup
import sys
if 'google.colab' in sys.modules:
    print("Running on Google Colab - installing packages...")
    !pip install gymnasium[classic-control,box2d,atari,accept-rom-license] matplotlib seaborn numpy pillow -q
    print("Installation complete!")

import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to {SEED}")

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym
from gymnasium import spaces
from typing import Tuple, Dict, Any, Optional
from collections import deque
import time

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"Gymnasium version: {gym.__version__}")
print("✓ All imports successful!")

## 1. Gymnasium API Deep Dive

Gymnasium (formerly OpenAI Gym) is the standard API for RL environments. Understanding this API is essential for all RL work.

### The Core API

Every Gymnasium environment implements this interface:

```python
env = gym.make('CartPole-v1')  # Create environment

# Reset environment to initial state
observation, info = env.reset(seed=42)

# Take action in environment
observation, reward, terminated, truncated, info = env.step(action)

# Close environment
env.close()
```

### Key Concepts

**Observation Space**: Defines the structure of observations
- `Box`: Continuous values (e.g., position, velocity)
- `Discrete`: Integer values (e.g., grid position)
- `MultiDiscrete`: Multiple discrete values
- `MultiBinary`: Binary values

**Action Space**: Defines available actions
- `Discrete(n)`: n discrete actions
- `Box`: Continuous actions (e.g., motor torques)

**Termination vs Truncation**:
- `terminated`: Episode ended naturally (goal reached, failure)
- `truncated`: Episode cut off artificially (time limit)
- `done = terminated or truncated`

In [ ]:
# Exploring environment spaces
print("=" * 70)
print("EXPLORING GYMNASIUM ENVIRONMENTS")
print("=" * 70 + "\n")

environments = [
    'CartPole-v1',
    'MountainCar-v0',
    'Pendulum-v1',
    'LunarLander-v2',
]

for env_name in environments:
    try:
        env = gym.make(env_name)
        
        print(f"Environment: {env_name}")
        print(f"  Observation Space: {env.observation_space}")
        print(f"  Action Space: {env.action_space}")
        
        # Sample random observation and action
        sample_obs = env.observation_space.sample()
        sample_action = env.action_space.sample()
        
        print(f"  Sample Observation: {sample_obs}")
        print(f"  Sample Action: {sample_action}")
        
        # Get observation and action dimensions
        if isinstance(env.observation_space, spaces.Box):
            obs_dim = env.observation_space.shape[0]
            print(f"  Observation Dimension: {obs_dim}")
        
        if isinstance(env.action_space, spaces.Discrete):
            n_actions = env.action_space.n
            print(f"  Number of Actions: {n_actions}")
        elif isinstance(env.action_space, spaces.Box):
            action_dim = env.action_space.shape[0]
            print(f"  Action Dimension: {action_dim}")
            print(f"  Action Bounds: [{env.action_space.low}, {env.action_space.high}]")
        
        env.close()
        print()
        
    except Exception as e:
        print(f"  Could not load {env_name}: {e}\n")

print("✓ Environment exploration complete!")

## 2. Exploring Built-in Environments

Gymnasium provides many pre-built environments for testing RL algorithms.

### Environment Categories

**Classic Control**:
- CartPole: Balance pole on cart
- MountainCar: Drive car up hill with momentum
- Pendulum: Swing pendulum upright
- Acrobot: Two-link robot arm

**Box2D**:
- LunarLander: Land spacecraft safely
- BipedalWalker: Teach robot to walk
- CarRacing: Drive car on track

**Atari** (requires ROM license):
- Pong, Breakout, Space Invaders, etc.
- High-dimensional visual observations

**MuJoCo** (requires license):
- Hopper, Walker, Humanoid, etc.
- Physics-based robotics simulation

In [ ]:
# Demonstrate complete episode
print("=" * 70)
print("RUNNING COMPLETE EPISODE: CartPole-v1")
print("=" * 70 + "\n")

env = gym.make('CartPole-v1')
observation, info = env.reset(seed=SEED)

episode_reward = 0
observations = []
actions = []
rewards = []

print(f"{'Step':<6} {'Action':<8} {'Reward':<8} {'Cart Pos':<12} {'Pole Angle':<12} {'Done'}")
print("-" * 70)

for step in range(200):
    # Random action
    action = env.action_space.sample()
    
    # Step environment
    next_observation, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    
    # Store data
    observations.append(observation)
    actions.append(action)
    rewards.append(reward)
    episode_reward += reward
    
    # Print (first 10 and last)
    if step < 10 or done:
        cart_pos = observation[0]
        pole_angle = observation[2]
        action_name = "RIGHT" if action == 1 else "LEFT"
        print(f"{step:<6} {action_name:<8} {reward:<8.1f} {cart_pos:<12.4f} {pole_angle:<12.4f} {done}")
    elif step == 10:
        print("  ...")
    
    observation = next_observation
    
    if done:
        break

env.close()

print("\n" + "=" * 70)
print(f"Episode finished after {step + 1} steps")
print(f"Total reward: {episode_reward}")
print("=" * 70)

## 3. Creating Custom Environments

You can create custom environments by subclassing `gym.Env` and implementing:
- `reset()`: Initialize environment
- `step(action)`: Execute action, return observation, reward, done, info
- Define `observation_space` and `action_space`

Let's build a custom GridWorld environment from scratch.

In [ ]:
class CustomGridWorld(gym.Env):
    """
    Custom GridWorld Environment.
    
    Agent navigates grid to reach goal while avoiding obstacles.
    
    Observation: (row, col) position
    Actions: 0=Up, 1=Down, 2=Left, 3=Right
    Rewards: -1 per step, +10 at goal, -5 for obstacle
    """
    
    def __init__(self, size: int = 5, obstacles: list = None):
        super().__init__()
        
        self.size = size
        self.start = (0, 0)
        self.goal = (size-1, size-1)
        self.obstacles = obstacles if obstacles else [(1, 1), (2, 2), (3, 1)]
        
        # Define spaces
        self.observation_space = spaces.Box(
            low=0, high=size-1, shape=(2,), dtype=np.int32
        )
        self.action_space = spaces.Discrete(4)  # Up, Down, Left, Right
        
        self.state = None
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array(self.start, dtype=np.int32)
        return self.state.copy(), {}
    
    def step(self, action):
        # Move based on action
        new_state = self.state.copy()
        
        if action == 0:  # Up
            new_state[0] = max(0, new_state[0] - 1)
        elif action == 1:  # Down
            new_state[0] = min(self.size - 1, new_state[0] + 1)
        elif action == 2:  # Left
            new_state[1] = max(0, new_state[1] - 1)
        elif action == 3:  # Right
            new_state[1] = min(self.size - 1, new_state[1] + 1)
        
        self.state = new_state
        
        # Check if goal reached
        if tuple(self.state) == self.goal:
            reward = 10.0
            terminated = True
        # Check if hit obstacle
        elif tuple(self.state) in self.obstacles:
            reward = -5.0
            terminated = False
        # Normal step
        else:
            reward = -1.0
            terminated = False
        
        truncated = False
        info = {}
        
        return self.state.copy(), reward, terminated, truncated, info
    
    def render(self):
        grid = np.zeros((self.size, self.size), dtype=str)
        grid[:, :] = '.'
        
        # Mark obstacles
        for obs in self.obstacles:
            grid[obs] = 'X'
        
        # Mark goal
        grid[self.goal] = 'G'
        
        # Mark agent
        grid[tuple(self.state)] = 'A'
        
        print("\nGrid (A=Agent, G=Goal, X=Obstacle):")
        for row in grid:
            print(' '.join(row))
        print()

# Test custom environment
print("Testing Custom GridWorld Environment...\n")

env = CustomGridWorld(size=5)
observation, info = env.reset(seed=SEED)

print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Initial observation: {observation}")

env.render()

# Take a few steps
actions = [3, 3, 1, 1, 3]  # Right, Right, Down, Down, Right
action_names = ['Up', 'Down', 'Left', 'Right']

for action in actions:
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Action: {action_names[action]} | Reward: {reward:+.1f} | Position: {obs} | Done: {terminated}")
    
    if terminated:
        print("\n✓ Goal reached!")
        break

env.render()

print("\n✓ Custom environment working correctly!")

## 4. Environment Wrappers

Wrappers modify environment behavior without changing the environment itself. This is a powerful design pattern in RL.

### Common Wrapper Types

1. **ObservationWrapper**: Modify observations
2. **RewardWrapper**: Modify rewards
3. **ActionWrapper**: Modify actions
4. **Wrapper**: Modify any aspect

### Why Wrappers?

- **Modularity**: Stack multiple modifications
- **Reusability**: Same wrapper works on any environment
- **Clean code**: Environment code stays unchanged

In [ ]:
# Example: Reward clipping wrapper
class ClipRewardWrapper(gym.RewardWrapper):
    """
    Clip rewards to [-clip_value, +clip_value].
    Useful for stabilizing training.
    """
    def __init__(self, env, clip_value=1.0):
        super().__init__(env)
        self.clip_value = clip_value
    
    def reward(self, reward):
        return np.clip(reward, -self.clip_value, self.clip_value)

# Example: Frame stacking wrapper (for Atari)
class FrameStackWrapper(gym.ObservationWrapper):
    """
    Stack last n observations.
    Gives agent temporal information.
    """
    def __init__(self, env, n_frames=4):
        super().__init__(env)
        self.n_frames = n_frames
        self.frames = deque(maxlen=n_frames)
        
        # Update observation space
        low = np.repeat(self.observation_space.low, n_frames, axis=0)
        high = np.repeat(self.observation_space.high, n_frames, axis=0)
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)
    
    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        for _ in range(self.n_frames):
            self.frames.append(obs)
        return self.observation(obs), info
    
    def observation(self, observation):
        self.frames.append(observation)
        return np.concatenate(list(self.frames), axis=0)

# Example: Episode statistics wrapper
class EpisodeStatsWrapper(gym.Wrapper):
    """
    Track episode statistics.
    """
    def __init__(self, env):
        super().__init__(env)
        self.episode_reward = 0
        self.episode_length = 0
        self.episode_rewards = []
        self.episode_lengths = []
    
    def reset(self, **kwargs):
        if self.episode_length > 0:
            self.episode_rewards.append(self.episode_reward)
            self.episode_lengths.append(self.episode_length)
        
        self.episode_reward = 0
        self.episode_length = 0
        return self.env.reset(**kwargs)
    
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.episode_reward += reward
        self.episode_length += 1
        
        # Add stats to info
        if terminated or truncated:
            info['episode'] = {
                'r': self.episode_reward,
                'l': self.episode_length
            }
        
        return obs, reward, terminated, truncated, info
    
    def get_episode_rewards(self):
        return self.episode_rewards
    
    def get_episode_lengths(self):
        return self.episode_lengths

# Demonstrate wrapper usage
print("Demonstrating Environment Wrappers...\n")

# Create base environment
env = gym.make('CartPole-v1')

# Wrap with statistics tracker
env = EpisodeStatsWrapper(env)

# Run a few episodes
for episode in range(3):
    obs, info = env.reset(seed=SEED + episode)
    done = False
    
    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
    
    if 'episode' in info:
        print(f"Episode {episode + 1}: Reward = {info['episode']['r']:.1f}, Length = {info['episode']['l']}")

env.close()

print("\n✓ Wrappers allow clean, modular environment modifications!")

## 5. Visualization and Monitoring

Effective visualization is crucial for understanding RL agent behavior and debugging training.

In [ ]:
# Training curve visualization
def plot_training_curves(rewards, window=100):
    """
    Plot episode rewards with moving average.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Raw rewards
    axes[0].plot(rewards, alpha=0.3, label='Episode Reward')
    if len(rewards) >= window:
        moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
        axes[0].plot(range(window-1, len(rewards)), moving_avg, 
                    label=f'MA({window})', linewidth=2)
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Reward')
    axes[0].set_title('Training Rewards')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Histogram
    axes[1].hist(rewards, bins=50, alpha=0.7, edgecolor='black')
    axes[1].axvline(np.mean(rewards), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {np.mean(rewards):.1f}')
    axes[1].set_xlabel('Reward')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Reward Distribution')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Generate sample data and plot
sample_rewards = np.random.randn(1000).cumsum() + np.linspace(0, 100, 1000)
sample_rewards += np.random.randn(1000) * 10

print("Example Training Curve Visualization:\n")
plot_training_curves(sample_rewards)

print("\n✓ Visualization helps monitor training progress!")

## 6. Debugging RL Agents

RL debugging is notoriously difficult. Here are common issues and solutions.

### Common Problems

1. **Agent not learning**:
   - Check learning rate (try 1e-3, 1e-4)
   - Verify reward scaling (normalize rewards)
   - Check exploration (epsilon too low/high)
   - Ensure gradient flow (check for NaNs)

2. **Training unstable**:
   - Use smaller learning rate
   - Clip rewards
   - Normalize observations
   - Use gradient clipping

3. **Performance plateaus**:
   - Increase exploration
   - Try different hyperparameters
   - Check if environment is fully observable
   - Use curriculum learning

4. **Training too slow**:
   - Vectorize environments (parallel rollouts)
   - Use GPU acceleration
   - Reduce network size
   - Optimize replay buffer

### Debugging Checklist

✅ Random seed set for reproducibility  
✅ Environment tested independently  
✅ Observations in expected range  
✅ Rewards properly scaled  
✅ Actions valid for environment  
✅ Network architecture appropriate  
✅ Hyperparameters reasonable  
✅ Logging and monitoring enabled  
✅ Baseline comparison (random policy)  

In [ ]:
# Debugging utilities
def debug_environment(env_name, n_episodes=10):
    """
    Run diagnostic tests on environment.
    """
    print(f"\n{'='*70}")
    print(f"DEBUGGING ENVIRONMENT: {env_name}")
    print(f"{'='*70}\n")
    
    env = gym.make(env_name)
    
    # Test 1: Space properties
    print("Test 1: Space Properties")
    print(f"  Observation space: {env.observation_space}")
    print(f"  Action space: {env.action_space}")
    
    # Test 2: Reset consistency
    print("\nTest 2: Reset Consistency")
    obs1, _ = env.reset(seed=42)
    obs2, _ = env.reset(seed=42)
    print(f"  Same seed → same obs: {np.allclose(obs1, obs2)}")
    
    # Test 3: Observation bounds
    print("\nTest 3: Observation Bounds")
    observations = []
    for _ in range(100):
        action = env.action_space.sample()
        obs, _, terminated, truncated, _ = env.step(action)
        observations.append(obs)
        if terminated or truncated:
            env.reset()
    
    observations = np.array(observations)
    print(f"  Observation min: {observations.min(axis=0)}")
    print(f"  Observation max: {observations.max(axis=0)}")
    print(f"  Observation mean: {observations.mean(axis=0)}")
    print(f"  Observation std: {observations.std(axis=0)}")
    
    # Test 4: Episode statistics
    print("\nTest 4: Episode Statistics (Random Policy)")
    episode_rewards = []
    episode_lengths = []
    
    for episode in range(n_episodes):
        obs, _ = env.reset(seed=42 + episode)
        episode_reward = 0
        episode_length = 0
        done = False
        
        while not done and episode_length < 1000:
            action = env.action_space.sample()
            obs, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward
            episode_length += 1
            done = terminated or truncated
        
        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
    
    print(f"  Mean reward: {np.mean(episode_rewards):.2f} ± {np.std(episode_rewards):.2f}")
    print(f"  Mean length: {np.mean(episode_lengths):.2f} ± {np.std(episode_lengths):.2f}")
    print(f"  Min/Max reward: {np.min(episode_rewards):.2f} / {np.max(episode_rewards):.2f}")
    
    env.close()
    
    print(f"\n{'='*70}")
    print("✓ Environment debugging complete!")
    print(f"{'='*70}\n")

# Run diagnostics
debug_environment('CartPole-v1', n_episodes=20)

## 7. Best Practices

Professional RL development follows these practices:

### 1. Reproducibility
```python
# Set ALL random seeds
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
env.reset(seed=SEED)
```

### 2. Logging
```python
# Log metrics every episode
import wandb  # or tensorboard

wandb.log({
    'episode_reward': reward,
    'episode_length': length,
    'epsilon': epsilon,
    'loss': loss,
})
```

### 3. Checkpointing
```python
# Save model regularly
if episode % 100 == 0:
    agent.save(f'checkpoint_episode_{episode}.pth')
```

### 4. Evaluation
```python
# Separate train/eval modes
def evaluate(agent, env, n_episodes=10):
    agent.eval()  # Disable exploration
    rewards = []
    for _ in range(n_episodes):
        # Run episode...
        rewards.append(episode_reward)
    return np.mean(rewards)
```

### 5. Hyperparameter Management
```python
# Use config files
config = {
    'lr': 0.001,
    'gamma': 0.99,
    'epsilon': 0.1,
    'batch_size': 64,
    ...
}
```

## 8. Summary

### What We Learned

1. **Gymnasium API**: The standard interface for RL environments
   - `reset()`, `step()`, `close()`
   - Observation and action spaces
   - Terminated vs truncated

2. **Custom Environments**: Build your own RL problems
   - Subclass `gym.Env`
   - Implement `reset()` and `step()`
   - Define spaces

3. **Wrappers**: Modify environments modularly
   - ObservationWrapper, RewardWrapper, ActionWrapper
   - Stack multiple wrappers
   - Clean, reusable code

4. **Visualization**: Monitor training effectively
   - Training curves
   - Episode statistics
   - Reward distributions

5. **Debugging**: Common issues and solutions
   - Agent not learning
   - Training instability
   - Performance plateaus

6. **Best Practices**: Professional RL development
   - Reproducibility
   - Logging
   - Checkpointing
   - Evaluation

### Key Takeaways

- **Gymnasium is the foundation** of all RL work - master the API
- **Wrappers are powerful** - use them to keep code clean
- **Debugging RL is hard** - follow systematic approaches
- **Visualization is essential** - always monitor training
- **Reproducibility matters** - set seeds, log everything

### Next Steps

Now that you understand the practical tools, you're ready for the mathematical foundations:

**Next Lesson**: [1a: Markov Decision Processes (MDPs)](./1a_mdp_theory.ipynb)
- Formal definition of MDPs
- Bellman equations
- Optimal policies and value functions

### Exercises

1. **Create a custom environment** for a problem you're interested in
2. **Build wrapper** that normalizes observations to [-1, 1]
3. **Debug** why a random policy doesn't work on your environment
4. **Visualize** episode trajectories in your custom environment
5. **Compare** performance of different exploration strategies

---

**Congratulations!** You now have the practical skills to work with RL environments professionally.

**Next**: [Lesson 1a: Markov Decision Processes →](./1a_mdp_theory.ipynb)